In [1]:
!pip install torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.6/138.6 MB 28.7 MB/s  0:00:04m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 22.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 27.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 10.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [torch]32m4/5 [torch]]x]


In [2]:
import os
# Αποφυγή σύγκρουσης των βιβλιοθηκών OpenMP στο Mac
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
# Χρήση ενός νήματος για αποφυγή κατάρρευσης του kernel σε macOS
torch.set_num_threads(1)

import torch.nn as nn
from torch.nn import functional as F

# ==============================================================================
# 1. ΥΠΕΡΠΑΡΑΜΕΤΡΟΙ (Ρυθμίσεις του Μοντέλου)
# ==============================================================================
block_size = 32        # Πόσα tokens κοιτάζει πίσω το μοντέλο (Context Length)
n_embd = 32            # Η διάσταση του διανύσματος κάθε λέξης/χαρακτήρα
n_head = 4             # Πόσες κεφαλές προσοχής λειτουργούν παράλληλα
n_layer = 4            # Πόσα μπλοκ Transformer θα στοιβάξουμε
learning_rate = 1e-3   # Ρυθμός μάθησης
max_iters = 1500       # Επαναλήψεις εκπαίδευσης
eval_interval = 300    # Έλεγχος σφάλματος (loss)

# Ορισμός σε απλό επεξεργαστή (CPU)
device = 'cpu'
torch.manual_seed(1337)

# ==============================================================================
# 2. ΤΟ ΣΥΝΟΛΟ ΔΕΔΟΜΕΝΩΝ (Dataset)
# ==============================================================================
text = """Μια φορά κι έναν καιρό ήταν ένας σοφός βασιλιάς.
Ο βασιλιάς ζούσε σε ένα μεγάλο πέτρινο κάστρο.
Μια φορά κι έναν καιρό ήταν μια καλή βασίλισσα.
Η βασίλισσα αγαπούσε όλα τα ζώα του δάσους.
Υπήρχε κάποτε ένας νεαρός πρίγκιπας στο κάστρο.
Ο πρίγκιπας είχε ένα μικρό άσπρο άλογο.
Υπήρχε κάποτε μια όμορφη πριγκίπισσα.
Η πριγκίπισσα διάβαζε παλιά βιβλία στη βιβλιοθήκη.
Ο βασιλιάς και η βασίλισσα κυβερνούσαν με δικαιοσύνη.
Το κάστρο ήταν χτισμένο πάνω σε έναν ψηλό λόφο.
Ένα μικρό κορίτσι επισκέφτηκε το παλάτι μια μέρα.
Ο πρίγκιπας και η πριγκίπισσα έπαιζαν συχνά στον κήπο.
"""

# ==============================================================================
# 3. Ο TOKENIZER (Από Χαρακτήρες σε Αριθμούς)
# ==============================================================================
chars = sorted(list(set(text)))
vocab_size = len(chars)

char_to_id = {ch: i for i, ch in enumerate(chars)}
id_to_char = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [char_to_id[c] for c in s]
decode = lambda l: ''.join([id_to_char[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)

def get_batch():
    ix = torch.randint(len(data) - block_size, (16,))
    x = torch.stack([data[i:i+block_size] for i in ix]).to(device)
    y = torch.stack([data[i+1:i+block_size+1] for i in ix]).to(device)
    return x, y

# ==============================================================================
# 4. Η ΑΡΧΙΤΕΚΤΟΝΙΚΗ TRANSFORMER
# ==============================================================================
class Head(nn.Module):
    """ Μία αυτόνομη κεφαλή αυτοπροσοχής (Self-Attention) """
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        
        wei = q @ k.transpose(-2, -1) * (C**-0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    """ Πολλαπλές κεφαλές προσοχής που τρέχουν παράλληλα """
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, n_embd)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        return out

class FeedForward(nn.Module):
    """ Δίκτυο δύο επιπέδων για αυτόνομη επεξεργασία κάθε θέσης """
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Ένα πλήρες δομικό μπλοκ Transformer """
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

# ==============================================================================
# 5. ΤΟ ΠΛΗΡΕΣ ΓΛΩΣΣΙΚΟ ΜΟΝΤΕΛΟ (Mini LLM)
# ==============================================================================
class MiniLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens, temperature=1.0):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# Αρχικοποίηση Μοντέλου
model = MiniLanguageModel().to(device)
param_count = sum(p.numel() for p in model.parameters())
print(f"Το μοντέλο δημιουργήθηκε επιτυχώς! Συνολικές παράμετροι: {param_count:,}")

# ==============================================================================
# 6. Ο ΒΡΟΧΟΣ ΕΚΠΑΙΔΕΥΣΗΣ (Training Loop)
# ==============================================================================
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

print("\n--- Έναρξη Εκπαίδευσης ---")
for iter in range(max_iters):
    xb, yb = get_batch()

    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if iter % eval_interval == 0 or iter == max_iters - 1:
        print(f"Βήμα {iter:4d} | Σφάλμα (Loss): {loss.item():.4f}")

# ==============================================================================
# 7. ΠΑΡΑΓΩΓΗ ΚΕΙΜΕΝΟΥ (Inference)
# ==============================================================================
print("\n--- Δοκιμή Παραγωγής Κειμένου ---")
prompt = "Μια φορά "
context = torch.tensor(encode(prompt), dtype=torch.long, device=device).unsqueeze(0)

generated_ids = model.generate(context, max_new_tokens=150, temperature=0.7)
print(decode(generated_ids[0].tolist()))

Το μοντέλο δημιουργήθηκε επιτυχώς! Συνολικές παράμετροι: 54,120

--- Έναρξη Εκπαίδευσης ---
Βήμα    0 | Σφάλμα (Loss): 3.8458
Βήμα  300 | Σφάλμα (Loss): 1.0757
Βήμα  600 | Σφάλμα (Loss): 0.2873
Βήμα  900 | Σφάλμα (Loss): 0.1804
Βήμα 1200 | Σφάλμα (Loss): 0.1509
Βήμα 1499 | Σφάλμα (Loss): 0.1385

--- Δοκιμή Παραγωγής Κειμένου ---
Μια φορά κι έναν καιρό ήταν μια καλή βασίλισσα.
Η βασίλισσα αγαπούσε όλα τα ζώα τα ζώα του δάσους.
Υπήρχε κάποτε ένας νεαρός πρίγκιπας στο κάστρο.
Ο πρίγκιπας 


In [3]:
# Δοκιμή 1: Χαμηλή θερμοκρασία (Πολύ αυστηρό/προβλέψιμο)
prompt = "Υπήρχε κάποτε "
context = torch.tensor(encode(prompt), dtype=torch.long, device=device).unsqueeze(0)

print("--- Temperature 0.2 (Συντηρητικό) ---")
out_low = model.generate(context, max_new_tokens=120, temperature=0.2)
print(decode(out_low[0].tolist()))

print("\n--- Temperature 1.8 (Υπερβολικά τυχαίο / ασυναρτησία) ---")
out_high = model.generate(context, max_new_tokens=120, temperature=1.8)
print(decode(out_high[0].tolist()))

--- Temperature 0.2 (Συντηρητικό) ---
Υπήρχε κάποτε μια όμορφη πριγκίπισσα.
Η πριγκίπισσα διάβαζε παλιά βιβλία στη βιβλιοθήκη.
Ο βασιλιάς και η βασίλισσα κυβερνούσαν με δικ

--- Temperature 1.8 (Υπερβολικά τυχαίο / ασυναρτησία) ---
Υπήρχε κάποτε μια όμορρφη πριγ.
Ο πριγκίπισσα άβαστη μιγοσμε πα.
Η πριγκΗ ό πονάστια  κέπριγο.


ζαροαρίγκιπας κοριγκς άσπαλι διαλι κυ
